# Bible Study Virtual Assistant — Class Project Demo

An LLM-based virtual assistant that answers Bible-study questions using **structured tools**, not just model memory.

**Models compared**
- Smaller: `microsoft/Phi-3.5-mini-instruct`
- Larger: `mistralai/Mistral-7B-Instruct-v0.3` (4-bit on Colab)

**Tools**
- Bible verse lookup + TF-IDF search (KJV, public domain)
- BEMA Discipleship podcast transcript retrieval (TF-IDF)
- DuckDuckGo web search for historical context

**Prompting techniques**: zero-shot, few-shot, chain-of-thought

**Security**: 5 prompt-injection / abuse tests

## 1. Setup (Colab)

Skip this cell if running locally with the repo already cloned.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/YOUR_USERNAME/bible-study-va.git"  # <- edit
REPO_DIR = "bible-study-va"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # local: assume notebook is run from notebooks/ inside the repo
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

print("cwd:", os.getcwd())

## 2. Build the data

- Download the KJV Bible JSON (~5 MB, one-time).
- Scrape a small batch of BEMA transcripts so the demo runs fast. Bump `--max-episodes` for the full corpus.

In [ ]:
!python scripts/load_bible.py
!python scripts/scrape_bema.py --max-episodes 15

## 3. Smoke-test the tools (no LLM yet)

In [ ]:
from src.bible_tool import BibleTool
from src.bema_tool import BemaTool

bible = BibleTool("data/bible/bible.json")
bema = BemaTool("data/bema/transcripts", "data/bema/episodes.json")

print("Verse lookup — John 3:16:")
for v in bible.lookup("John 3:16"):
    print(" ", v)

print("\nVerse search — 'babylon':")
for hit in bible.search("babylon great fallen", k=3):
    print(f"  ({hit.score:.2f}) {hit.item}")

print(f"\nBEMA chunks indexed: {len(bema.chunks)}")
print("BEMA search — 'creation story':")
for hit in bema.search("creation story Genesis Eastern thinking", k=2):
    print(f"  ({hit.score:.2f}) {hit.item}")

## 4. Load both LLMs

On Colab T4: Phi-3.5 in fp16 (~7 GB), Mistral-7B in 4-bit (~5 GB). To save memory you can load them one at a time and re-run the eval cell.

In [ ]:
from src.llm import LLM

phi3 = LLM.load("phi3")
mistral = LLM.load("mistral")

## 5. Build the agent and answer a sample question

In [ ]:
from src.agent import Agent

agent = Agent(
    bible_path="data/bible/bible.json",
    bema_transcripts_dir="data/bema/transcripts",
    bema_episodes_json="data/bema/episodes.json",
    enable_web=True,
)

demo = agent.answer(
    "What are hidden meanings and symbols behind Babylon in the Bible?",
    model=phi3,
    technique="zero_shot",
)

print("ROUTING:", demo.routing.reason)
print(f"LATENCY: {demo.latency_s:.1f}s | tokens in/out: {demo.tokens_in}/{demo.tokens_out}")
print("\nANSWER:\n", demo.text)
print("\nSOURCES:")
for s in demo.sources[:5]:
    print(" -", s.label)

## 6. Model + prompting-technique comparison

Run our 5 example questions through both models and all three techniques, then collect a results table.

In [ ]:
import pandas as pd

QUESTIONS = [
    "What are hidden meanings and symbols behind Babylon in the Bible?",
    "What was the common Jewish cultural acceptance of wine?",
    "How did the Roman Empire clash with the Jews?",
    "What is a chiasm and what are examples in Deuteronomy?",
    "What is the difference between a Pharisee and a teacher of the law?",
]

TECHNIQUES = ["zero_shot", "few_shot", "chain_of_thought"]
MODELS = {"phi3": phi3, "mistral": mistral}

rows = []
for q in QUESTIONS:
    for mname, m in MODELS.items():
        for t in TECHNIQUES:
            ans = agent.answer(q, model=m, technique=t, max_new_tokens=350)
            rows.append({
                "question": q[:60] + "...",
                "model": mname,
                "technique": t,
                "latency_s": round(ans.latency_s, 2),
                "tokens_out": ans.tokens_out,
                "answer": ans.text[:300] + "...",
            })

df = pd.DataFrame(rows)
df

In [ ]:
import matplotlib.pyplot as plt

agg = df.groupby(["model", "technique"])["latency_s"].mean().unstack()
ax = agg.plot(kind="bar", figsize=(8, 4))
ax.set_title("Mean response time by model and prompting technique")
ax.set_ylabel("seconds")
ax.set_xlabel("model")
plt.tight_layout()
plt.show()

## 7. Manual quality scoring

After reading the answers in the table above, fill in a 1–5 quality score for each row in the cell below. This is the qualitative half of the rubric.

In [ ]:
df["quality"] = 0  # <-- edit per row, e.g. df.loc[0, 'quality'] = 4
df.groupby(["model", "technique"])["quality"].mean().unstack()

## 8. Prompt-injection / security tests

Run all 5 attacks against the (smaller, faster) model and see whether each is defended.

In [ ]:
from src.security import run_all

results = run_all(agent, phi3, technique="zero_shot")
for r in results:
    print(r)
    print("-" * 80)

successes = sum(r.succeeded for r in results)
print(f"\nDefense summary: {len(results) - successes}/{len(results)} attacks blocked")

## 9. Discussion / limitations

Things to write up in the report:

- Where Phi-3.5 was clearly weaker / stronger than Mistral-7B
- How chain-of-thought changed answer quality vs latency
- Which attack(s) succeeded and what an additional defense would look like
  (input filtering, output filtering, separating data from instructions, etc.)
- TF-IDF vs embeddings: cases where TF-IDF missed clearly-relevant context